# Phase 8 — Jacobian IK Math

## 1. Forward velocity kinematics: x_dot = J(q) q_dot

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

import numpy as np
import mujoco

# Load KR6 model
MODEL_PATH = str(pathlib.Path("../models/kr6.xml").resolve())
model = mujoco.MjModel.from_xml_path(MODEL_PATH)
data  = mujoco.MjData(model)

# Set a non-trivial joint configuration
q0 = np.array([0.3, -0.5, 0.8, 0.2, -0.4, 0.6])
data.qpos[:6] = q0
mujoco.mj_forward(model, data)

# Compute geometric Jacobian for link6
body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "link6")
jacp = np.zeros((3, model.nv))   # linear  velocity rows
jacr = np.zeros((3, model.nv))   # angular velocity rows
mujoco.mj_jacBody(model, data, jacp, jacr, body_id)

# Stack into 6×6 (first 6 DOF only)
J = np.vstack([jacp[:, :6], jacr[:, :6]])

print("q =", np.round(q0, 3))
print("\nJacobian J (6×6):")
print(np.round(J, 4))
print("\nshape:", J.shape)


## 2. Naive inverse — why it fails near singularities

In [ ]:
print("Condition number at q0:", round(np.linalg.cond(J), 2))

# Drive arm toward a near-singular pose (fully extended, A2+A3 ≈ 0)
q_singular = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
data.qpos[:6] = q_singular
mujoco.mj_forward(model, data)
mujoco.mj_jacBody(model, data, jacp, jacr, body_id)
J_sing = np.vstack([jacp[:, :6], jacr[:, :6]])
cond_sing = np.linalg.cond(J_sing)

print("Condition number at home (near-singular):", round(cond_sing, 2))

# Naive inverse  q_dot = inv(J) @ x_dot  — show why it explodes
x_dot_test = np.array([0.1, 0.0, 0.0, 0.0, 0.0, 0.0])
try:
    q_dot_naive = np.linalg.solve(J_sing, x_dot_test)
    print("\nNaive q_dot:", np.round(q_dot_naive, 4))
    print("Max |q_dot|:", round(float(np.max(np.abs(q_dot_naive))), 4),
          "<-- can be huge near singularity")
except np.linalg.LinAlgError as e:
    print("Singular matrix — solve failed:", e)


## 3. Damped least squares: q_dot = J^T inv(J J^T + lambda^2 I) x_dot

In [ ]:
def damped_least_squares_ik(J, x_dot, lambda_sq=0.01):
    """
    q_dot = J^T @ inv(J J^T + lambda^2 * I) @ x_dot
    lambda_sq: damping factor squared. Larger = safer near singularity.
    """
    I6    = np.eye(J.shape[0])
    A     = J @ J.T + lambda_sq * I6
    q_dot = J.T @ np.linalg.solve(A, x_dot)
    return q_dot

# Compare naive vs DLS on the near-singular pose
print("=== Near-singular pose (home) ===")
print("x_dot:", x_dot_test)

try:
    q_naive = np.linalg.solve(J_sing, x_dot_test)
    print("Naive  q_dot:", np.round(q_naive, 4), "  |max|:", round(float(np.max(np.abs(q_naive))), 4))
except np.linalg.LinAlgError:
    print("Naive: SINGULAR — failed")

q_dls = damped_least_squares_ik(J_sing, x_dot_test, lambda_sq=0.01)
print("DLS    q_dot:", np.round(q_dls,   4), "  |max|:", round(float(np.max(np.abs(q_dls))),   4))

print("\n=== Well-conditioned pose (q0) ===")
q_dls0 = damped_least_squares_ik(J, x_dot_test, lambda_sq=0.01)
print("DLS    q_dot:", np.round(q_dls0, 4))


## 4. Worked example

In [ ]:
# Worked example: from q0, move EE +5 cm in X over 0.5 s
data.qpos[:6] = q0
mujoco.mj_forward(model, data)
mujoco.mj_jacBody(model, data, jacp, jacr, body_id)
J_ex = np.vstack([jacp[:, :6], jacr[:, :6]])

x_dot_ex = np.array([0.1, 0.0, 0.0,   # linear  vx=0.1 m/s
                      0.0, 0.0, 0.0])  # angular none

q_dot_ex = damped_least_squares_ik(J_ex, x_dot_ex, lambda_sq=0.01)

print("Starting q:", np.round(q0, 3))
print("Desired x_dot [m/s]:", x_dot_ex)
print("Solved  q_dot [rad/s]:", np.round(q_dot_ex, 4))

# Verify: J @ q_dot ≈ x_dot
x_dot_check = J_ex @ q_dot_ex
print("\nVerification  J @ q_dot =", np.round(x_dot_check, 4))
print("vs desired x_dot         =", x_dot_ex)
print("Error:", np.round(x_dot_check - x_dot_ex, 6))
